# Table 1: Execution Latency

Server encryption (`grt_server.py`, Python) and client decryption (`igrt_client.js` `descrambleBuffer32IGRT`, Node.js V8). Edit the configuration cell to change images, grids, or run counts. Run all cells from the `tests_2/` folder.

In [4]:
import json
import random
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
from PIL import Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src" / "cryptography"))
import grt_server

# ----- Configuration: edit this block to change images, grids, or run counts -----
SCENARIOS = [
    {"name": "SD 480p",   "file": "docs/images/t1_SD_480p.png",   "M": 30, "N": 25},
    {"name": "FHD 1080p", "file": "docs/images/t2_FHD_1080p.png", "M": 40, "N": 40},
    {"name": "HD 720p",   "file": "docs/images/t5_HD_720p.png",   "M": 36, "N": 40},
]
WARMUP, ITERS, SEED = 5, 30, 2026
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


def make_keys(M, N):
    """Reproducible valid GRT keys (a, b, c, d, rm, rn)."""
    random.seed(SEED)
    np.random.seed(SEED)
    keys = tuple(int(k) for k in grt_server.generate_valid_grt_keys(M, N))
    ok, msg = grt_server.validate_grt_parameters(M, N, *keys)
    assert ok, msg
    return keys


def load_scenario(sc):
    """Load image, pad (edge replication) to a multiple of the grid, return padded image and original size."""
    img = Image.open(ROOT / sc["file"]).convert("RGB")
    padded, orig_size = grt_server.pad_image_to_grid(img, sc["M"], sc["N"])
    return padded, orig_size


def stats(values):
    arr = np.array(values)
    return {"mean": float(arr.mean()), "std": float(arr.std()), "median": float(np.median(arr))}

In [5]:
IGRT_JS = ROOT / "src" / "cryptography" / "igrt_client.js"

JS_RUNNER = """
const fs = require('fs');
const igrt = require(process.argv[2]);
const [inPath, outPath, w, h, M, N, a, b, c, d, rm, rn, iters, warm] =
    [process.argv[3], process.argv[4], ...process.argv.slice(5).map(Number)];
const src32 = new Uint32Array(new Uint8Array(fs.readFileSync(inPath)).buffer);
const dst32 = new Uint32Array(src32.length);
for (let i = 0; i < warm; i++) igrt.descrambleBuffer32IGRT(src32, dst32, w, h, M, N, a, b, c, d, rm, rn);
const times = [];
for (let i = 0; i < iters; i++) {
    const t0 = process.hrtime.bigint();
    igrt.descrambleBuffer32IGRT(src32, dst32, w, h, M, N, a, b, c, d, rm, rn);
    times.push(Number(process.hrtime.bigint() - t0) / 1e6);
}
if (outPath !== '-') fs.writeFileSync(outPath, Buffer.from(dst32.buffer));
console.log(JSON.stringify(times));
"""


def run_igrt_client(scrambled, M, N, keys, iters=ITERS, warm=WARMUP, want_image=False):
    """Run the real igrt_client.js (descrambleBuffer32IGRT) in Node.js. Returns (timings_ms, restored_image|None)."""
    w, h = scrambled.size
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        (tmp / "in.bin").write_bytes(scrambled.convert("RGBA").tobytes())
        (tmp / "run.js").write_text(JS_RUNNER, encoding="utf-8")
        out_path = str(tmp / "out.bin") if want_image else "-"
        cmd = ["node", str(tmp / "run.js"), str(IGRT_JS).replace("\\", "/"), str(tmp / "in.bin"), out_path,
               str(w), str(h), str(M), str(N), *map(str, keys), str(iters), str(warm)]
        res = subprocess.run(cmd, capture_output=True, text=True, check=True)
        timings = json.loads(res.stdout.strip().splitlines()[-1])
        restored = None
        if want_image:
            restored = Image.frombytes("RGBA", (w, h), (tmp / "out.bin").read_bytes()).convert("RGB")
    return timings, restored

In [6]:
rows = []
for sc in SCENARIOS:
    M, N = sc["M"], sc["N"]
    img, orig_size = load_scenario(sc)
    keys = make_keys(M, N)

    for _ in range(WARMUP):
        grt_server.scramble_image_grt(img, M, N, *keys)
    enc = []
    for _ in range(ITERS):
        t0 = time.perf_counter()
        scrambled = grt_server.scramble_image_grt(img, M, N, *keys)
        enc.append((time.perf_counter() - t0) * 1000.0)

    dec, _ = run_igrt_client(scrambled, M, N, keys)
    e, d = stats(enc), stats(dec)
    rows.append({
        "scenario": sc["name"], "orig_size": list(orig_size), "padded_size": list(img.size),
        "grid": [M, N], "tiles": M * N, "tile_size": [img.width // M, img.height // N], "keys": list(keys),
        "server_enc_ms": e, "client_dec_ms": d,
        "roundtrip_ms": e["mean"] + d["mean"],
    })

(RESULTS_DIR / "latency.json").write_text(json.dumps(rows, indent=2))
print(f"{'Scenario':<10} {'Orig':>11} {'Padded':>11} {'Grid':>7} {'Tiles':>6} {'Tile':>8} {'Server Enc (ms)':>18} {'Client Dec (ms)':>18} {'Roundtrip':>10}")
for r in rows:
    print(f"{r['scenario']:<10} {r['orig_size'][0]:>5}x{r['orig_size'][1]:<5} {r['padded_size'][0]:>5}x{r['padded_size'][1]:<5} "
          f"{r['grid'][0]:>3}x{r['grid'][1]:<3} {r['tiles']:>6} {r['tile_size'][0]:>3}x{r['tile_size'][1]:<4} "
          f"{r['server_enc_ms']['mean']:>9.2f} +- {r['server_enc_ms']['std']:<5.2f} "
          f"{r['client_dec_ms']['mean']:>9.2f} +- {r['client_dec_ms']['std']:<5.2f} {r['roundtrip_ms']:>10.2f}")

Scenario          Orig      Padded    Grid  Tiles     Tile    Server Enc (ms)    Client Dec (ms)  Roundtrip
SD 480p      480x854     480x875    30x25     750  16x35        3.27 +- 0.30       0.57 +- 0.11        3.84
FHD 1080p   1920x1080   1920x1080   40x40    1600  48x27       12.60 +- 0.45       2.74 +- 0.19       15.34
HD 720p      720x1280    720x1280   36x40    1440  20x32        7.03 +- 0.28       1.24 +- 0.07        8.28
